In [ ]:
!pip install transformers datasets librosa torch

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from sklearn.manifold import TSNE

from transformers import (AutoModelForAudioClassification, AutoFeatureExtractor)

from datasets import load_dataset, Audio, DatasetDict

In [ ]:
model_or = AutoModelForAudioClassification.from_pretrained("openai/whisper-base")
model_ft = AutoModelForAudioClassification.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

feature_extractor = AutoFeatureExtractor.from_pretrained("VasilisAsim/whisper-finetuned-TESS")

In [ ]:
#Φόρτωση Dataset και speaker eindependent split
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})

dataset = load_tess_speaker_independent()
test_dataset = dataset["test"]

In [ ]:
embeddings = []
labels = []

num_samples = len(test_dataset)

In [ ]:
for i in range(num_samples):
    sample = test_dataset[i]
    audio = sample["audio"]["array"]
    label = sample["emotion"]
    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")

    with torch.no_grad():
        outputs = model_or.encoder(inputs["input_features"]) # Corrected line

    hidden_states = outputs.last_hidden_state
    pooled_embedding = hidden_states.mean(dim=1)
    embeddings.append(pooled_embedding.squeeze().numpy())

    labels.append(label)

In [ ]:
embeddings = np.array(embeddings)
labels = np.array(labels)

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

In [ ]:
plt.figure(figsize=(10,8))

for label in np.unique(labels):
    idx = labels == label
    plt.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1], label=label)

plt.legend()
plt.title("t-SNE Visualization of Emotion Embeddings")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.show()

In [ ]:
embeddings = []
labels = []

In [ ]:
#Αποθήκευση ετικετών και στοιχείων κάθε δείγματος στις List
for i in range(num_samples):
    sample = test_dataset[i]
    audio = sample["audio"]["array"]
    label = sample["emotion"]
    inputs = feature_extractor(audio, sampling_rate=16000, return_tensors="pt")

    with torch.no_grad():
        outputs = model_ft.encoder(inputs["input_features"])

    hidden_states = outputs.last_hidden_state
    pooled_embedding = hidden_states.mean(dim=1)
    embeddings.append(pooled_embedding.squeeze().numpy())

    labels.append(label)

In [ ]:
#Μετατροπή σε arrays
embeddings = np.array(embeddings)
labels = np.array(labels)

In [ ]:
#Δημιουργία T-SNE αντικειμένου και εφαρμογή του στα δείγματα
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
embeddings_2d = tsne.fit_transform(embeddings)

In [ ]:
plt.figure(figsize=(10,8))

for label in np.unique(labels):
    idx = labels == label
    plt.scatter(embeddings_2d[idx, 0], embeddings_2d[idx, 1], label=label)

plt.legend()
plt.title("t-SNE Visualization of Emotion Embeddings")
plt.xlabel("t-SNE Dimension 1")
plt.ylabel("t-SNE Dimension 2")
plt.show()